# ML Coding Practice - 1

Use this file for practicing ML coding. We may have additional files as these grow long. Have headers for each concept and implementation to keep it accessible in the future.

In [1]:
# Common imports
import torch
import torch.nn as nn
import torch.nn.functional as F

## Multihead Self-Attention (MHA)

One of the most important pieces for the transformer model

In [14]:
# Write the canoncial class for this.
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads, max_seq_len = 512):
        super().__init__()

        assert d_model % num_heads == 0
        self.head_dim = d_model // num_heads
        self.num_heads = num_heads

        # Define the layers.
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # Out proj.
        self.W_out = nn.Linear(d_model, d_model, bias=False)

        # Precompute the mask, register_buffer so it moves with .to(device).
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("mask", mask)

    def forward(self, x):
        # Batch, seq_len and dims (channels).
        B, T, C = x.shape

        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        # Break it up into multiple heads.
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, nh, T, hd)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, nh, T, hd)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, nh, T, hd)

        # Compute scaled-dot product attention.
        att = q @ k.transpose(-2,-1) * (self.head_dim**-0.5)      # (B, nh, T, T)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        # Combine.
        scores = att @ v      # (B, nh, T, hd)
        scores = scores.transpose(1, 2).contiguous().view(B, T, C)

        out = self.W_out(scores)
        return out


In [15]:
# --- test ---
B, T, C, n_heads = 2, 8, 64, 4
x = torch.randn(B, T, C)
mha = MultiHeadAttention(d_model=C, num_heads=n_heads, max_seq_len=T)

out = mha(x)
print(out.shape)

torch.Size([2, 8, 64])


In [16]:
x[1,1]

tensor([ 9.2535e-01, -9.3458e-01, -1.6440e-01,  1.5903e-01, -5.7435e-01,
        -3.2664e-02,  1.1358e+00,  8.5657e-01, -1.2135e+00, -7.5613e-02,
        -1.0249e-01, -1.8307e-01, -1.7937e+00, -1.3428e+00, -1.9259e-01,
        -6.6252e-02,  1.1320e+00, -7.4184e-01,  2.1568e-02, -1.7669e-03,
        -1.7552e-01, -5.2361e-01,  2.0473e-01, -4.4804e-01,  1.2041e+00,
        -3.7157e-01, -8.7688e-01, -7.1908e-01, -1.0752e+00, -2.6802e-01,
         1.7402e+00,  5.9872e-01,  1.0789e+00, -6.6952e-01,  2.5559e-01,
         1.3139e+00,  1.6465e+00, -7.7611e-01,  4.8159e-02, -5.5650e-01,
        -7.9127e-01, -2.1960e+00,  6.1267e-02, -1.0544e+00,  4.8984e-01,
         7.2099e-01, -1.7808e-01, -6.1071e-01, -1.4340e+00,  8.6040e-01,
         1.7835e+00, -1.0374e+00,  8.4181e-01,  2.9719e-01,  1.3131e+00,
         1.4437e+00, -2.8265e-01,  9.1696e-01, -2.5368e+00, -1.4426e+00,
        -1.3453e+00,  4.0347e-01,  5.3770e-02, -7.7269e-01])

In [18]:
# --- Test against nn.MultiheadAttention ---
torch.manual_seed(21)
B, T, C, n_heads = 2, 8, 64, 4
x = torch.randn(B, T, C)

mine = MultiHeadAttention(d_model=C, num_heads=n_heads, max_seq_len=T)

ref = nn.MultiheadAttention(embed_dim=C, num_heads=n_heads, bias=False, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mine.W_q.weight, mine.W_k.weight, mine.W_v.weight], dim=0))
    ref.out_proj.weight.copy_(mine.W_out.weight)

causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # True = masked out

out_mine = mine(x)
out_ref, _ = ref(x, x, x, attn_mask=causal_mask, need_weights=False)

print(torch.allclose(out_mine, out_ref, atol=1e-5))
print((out_mine - out_ref).abs().max())

True
tensor(1.1921e-07, grad_fn=<MaxBackward1>)
